# 论文 22：神经语言模型的缩放定律
## Jared Kaplan 等（2020）

### 可预测的缩放：损失是算力、数据量和参数量的函数

通过实证分析展示神经网络缩放过程中的幂律关系。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

np.random.seed(42)

## 缩放定律的表达式

主要发现：损失遵循幂律：
$$L(N) = \left(\frac{N_c}{N}\right)^{\alpha_N}$$

其中：
- N = 参数数量
- D = 数据集大小
- C = 计算预算（FLOPs）


In [ ]:
def power_law(x, a, b, c):
    '幂律: y = a * x^(-b) + c'
    return a * np.power(x, -b) + c

def scaling_law_params(x, a, b):
    'Simplified: L = a * N^(-b)'
    return a * np.power(x, -b)

# 理论标度定律常数（来自论文）
# 这些是 Kaplan 等人的近似值。
alpha_N = 0.076  # 参数缩放指数
alpha_D = 0.095  # 数据缩放指数
alpha_C = 0.050  # 计算缩放指数

N_c = 8.8e13     # 关键参数计数
D_c = 5.4e13     # 关键数据集大小
C_c = 3.1e8      # 关键计算

print("Scaling Law Parameters (from paper):")
print(f"  α_N (params): {alpha_N}")
print(f"  α_D (data): {alpha_D}")
print(f"  α_C (compute): {alpha_C}")

## 模拟不同尺度的模型训练

In [ ]:
class SimpleLanguageModel:
    '用于演示缩放行为的玩具语言模型'
    def __init__(self, num_params, vocab_size=100, embed_dim=32):
        self.num_params = num_params
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        
        # 根据参数计数计算容量
        self.capacity = np.log(num_params) / 10.0
    
    def train(self, dataset_size, num_steps):
        """模拟训练并返回最终损失
        
        损失会随着以下因素而减小：
        - 更多参数（更多容量）
        - 更多数据（更好的学习）
        - 更多训练（收敛）"""
        # 基础损失（词汇困惑）
        base_loss = np.log(self.vocab_size)
        
        # 参数缩放（更多参数=更低损失）
        param_factor = 1.0 / (1.0 + self.capacity)
        
        # 数据扩展（更多数据=更低损失）
        data_factor = 1.0 / (1.0 + np.log(dataset_size) / 15.0)
        
        # 训练收敛
        train_factor = np.exp(-num_steps / 1000.0)
        
        # 合并各项因素并加入噪声
        loss = base_loss * param_factor * data_factor * (0.5 + 0.5 * train_factor)
        loss += np.random.randn() * 0.05  # 添加噪声
        
        return max(loss, 1.0)  # 下限 1.0

print("Simple Language Model for scaling experiments")

## 实验 1：随模型大小（参数）缩放

In [ ]:
# 固定数据集和训练
dataset_size = 100000
num_steps = 1000

# 不同模型规模
param_counts = np.array([1e3, 5e3, 1e4, 5e4, 1e5, 5e5, 1e6, 5e6, 1e7])
losses_by_params = []

for N in param_counts:
    model = SimpleLanguageModel(num_params=int(N))
    loss = model.train(dataset_size, num_steps)
    losses_by_params.append(loss)

losses_by_params = np.array(losses_by_params)

# 拟合幂律
params_fit, _ = curve_fit(scaling_law_params, param_counts, losses_by_params)
a_params, b_params = params_fit

# 绘图
plt.figure(figsize=(10, 6))
plt.loglog(param_counts, losses_by_params, 'o', markersize=10, label='Measured Loss')
plt.loglog(param_counts, scaling_law_params(param_counts, *params_fit), 
           '--', linewidth=2, label=f'Power Law Fit: L ∝ N^{-b_params:.3f}')
plt.xlabel('Number of Parameters (N)')
plt.ylabel('Loss (L)')
plt.title('Scaling Law: Loss vs Model Size')
plt.legend()
plt.grid(True, alpha=0.3, which='both')
plt.show()

print(f"\nParameter Scaling:")
print(f"  Fitted exponent: {b_params:.4f}")
print(f"  Interpretation: Doubling params reduces loss by {(1 - 2**(-b_params))*100:.1f}%")

## 实验 2：随数据集大小缩放

In [ ]:
# 固定模型大小和训练
num_params = 1e6
num_steps = 1000

# 改变数据集大小
dataset_sizes = np.array([1e3, 5e3, 1e4, 5e4, 1e5, 5e5, 1e6, 5e6, 1e7])
losses_by_data = []

for D in dataset_sizes:
    model = SimpleLanguageModel(num_params=int(num_params))
    loss = model.train(int(D), num_steps)
    losses_by_data.append(loss)

losses_by_data = np.array(losses_by_data)

# 拟合幂律
data_fit, _ = curve_fit(scaling_law_params, dataset_sizes, losses_by_data)
a_data, b_data = data_fit

# 绘图
plt.figure(figsize=(10, 6))
plt.loglog(dataset_sizes, losses_by_data, 's', markersize=10, 
           color='orange', label='Measured Loss')
plt.loglog(dataset_sizes, scaling_law_params(dataset_sizes, *data_fit), 
           '--', linewidth=2, color='red', label=f'Power Law Fit: L ∝ D^{-b_data:.3f}')
plt.xlabel('Dataset Size (D)')
plt.ylabel('Loss (L)')
plt.title('Scaling Law: Loss vs Dataset Size')
plt.legend()
plt.grid(True, alpha=0.3, which='both')
plt.show()

print(f"\nDataset Scaling:")
print(f"  Fitted exponent: {b_data:.4f}")
print(f"  Interpretation: Doubling data reduces loss by {(1 - 2**(-b_data))*100:.1f}%")

## 实验 3：计算最优训练

Chinchilla 的发现：对于给定的计算预算，模型规模与数据量应当同步扩展。


In [ ]:
# 计算预算（任意单位）
compute_budgets = np.array([1e6, 5e6, 1e7, 5e7, 1e8, 5e8, 1e9])

# 对于每个计算预算，找到最佳的 N 和 D 分配
optimal_results = []

for C in compute_budgets:
    # Chinchilla：N 和 D 应该与计算同等扩展
    # C ≈ 6 * N * D（每个令牌每个参数 6 次 FLOP）
    # 最佳：N ∝ C^0.5，D ∝ C^0.5
    
    N_opt = int(np.sqrt(C / 6))
    D_opt = int(np.sqrt(C / 6))
    
    model = SimpleLanguageModel(num_params=N_opt)
    loss = model.train(D_opt, num_steps=1000)
    
    optimal_results.append({
        'compute': C,
        'params': N_opt,
        'data': D_opt,
        'loss': loss
    })

compute_vals = [r['compute'] for r in optimal_results]
losses_optimal = [r['loss'] for r in optimal_results]

# 拟合
compute_fit, _ = curve_fit(scaling_law_params, compute_vals, losses_optimal)
a_compute, b_compute = compute_fit

# 绘图
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 损失与计算
ax1.loglog(compute_vals, losses_optimal, '^', markersize=10, 
           color='green', label='Measured Loss')
ax1.loglog(compute_vals, scaling_law_params(compute_vals, *compute_fit), 
           '--', linewidth=2, color='darkgreen', 
           label=f'Power Law Fit: L ∝ C^{-b_compute:.3f}')
ax1.set_xlabel('Compute Budget (C)')
ax1.set_ylabel('Loss (L)')
ax1.set_title('Scaling Law: Loss vs Compute (Optimal Allocation)')
ax1.legend()
ax1.grid(True, alpha=0.3, which='both')

# 最佳 N 和 D 与计算
params_vals = [r['params'] for r in optimal_results]
data_vals = [r['data'] for r in optimal_results]

ax2.loglog(compute_vals, params_vals, 'o-', label='Optimal N (params)', linewidth=2)
ax2.loglog(compute_vals, data_vals, 's-', label='Optimal D (data)', linewidth=2)
ax2.set_xlabel('Compute Budget (C)')
ax2.set_ylabel('N or D')
ax2.set_title('Compute-Optimal Scaling: N ∝ C^0.5, D ∝ C^0.5')
ax2.legend()
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print(f"\nCompute-Optimal Scaling:")
print(f"  Loss exponent: {b_compute:.4f}")
print(f"  For 10x more compute, loss reduces by {(1 - 10**(-b_compute))*100:.1f}%")
print(f"\n  Chinchilla insight: Scale model AND data together!")
print(f"  N_optimal ∝ C^0.5")
print(f"  D_optimal ∝ C^0.5")

## 比较：不同的扩展策略

In [ ]:
# 比较相同计算预算的策略
C = 1e8

# 策略一：大模型，小数据
N_large = int(C / 1000)
D_small = 1000
model_large = SimpleLanguageModel(num_params=N_large)
loss_large_model = model_large.train(D_small, 1000)

# 策略2：小模型，大数据
N_small = 1000
D_large = int(C / 1000)
model_small = SimpleLanguageModel(num_params=N_small)
loss_small_model = model_small.train(D_large, 1000)

# 策略3：平衡（Chinchilla）
N_balanced = int(np.sqrt(C / 6))
D_balanced = int(np.sqrt(C / 6))
model_balanced = SimpleLanguageModel(num_params=N_balanced)
loss_balanced = model_balanced.train(D_balanced, 1000)

# 可视化
strategies = ['Large Model\nSmall Data', 'Small Model\nLarge Data', 'Balanced\n(Chinchilla)']
losses = [loss_large_model, loss_small_model, loss_balanced]
colors = ['red', 'orange', 'green']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 损失比较
ax1.bar(strategies, losses, color=colors, alpha=0.7)
ax1.set_ylabel('Final Loss')
ax1.set_title(f'Training Strategies (Same Compute Budget: {C:.0e})')
ax1.grid(True, alpha=0.3, axis='y')

# 资源分配
x = np.arange(3)
width = 0.35

params = [N_large, N_small, N_balanced]
data = [D_small, D_large, D_balanced]

ax2.bar(x - width/2, np.log10(params), width, label='log₁₀(Params)', alpha=0.7)
ax2.bar(x + width/2, np.log10(data), width, label='log₁₀(Data)', alpha=0.7)
ax2.set_ylabel('log₁₀(Count)')
ax2.set_title('Resource Allocation')
ax2.set_xticks(x)
ax2.set_xticklabels(strategies)
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nStrategy Comparison (Compute = {C:.0e}):")
print(f"\n1. Large Model (N={N_large:.0e}), Small Data (D={D_small:.0e}):")
print(f"   Loss = {loss_large_model:.4f}")
print(f"\n2. Small Model (N={N_small:.0e}), Large Data (D={D_large:.0e}):")
print(f"   Loss = {loss_small_model:.4f}")
print(f"\n3. Balanced (N={N_balanced:.0e}), (D={D_balanced:.0e}):")
print(f"   Loss = {loss_balanced:.4f} ← BEST")
print(f"\nKey Insight: Balanced scaling is compute-optimal!")

## 外推法：预测更大的模型

In [ ]:
# 使用拟合缩放定律来预测未来模型的性能
future_params = np.array([1e8, 1e9, 1e10, 1e11, 1e12])  # 100M 至 1T 参数
predicted_losses = scaling_law_params(future_params, *params_fit)

# 绘制外推结果
plt.figure(figsize=(12, 6))

# 已测量的数据
plt.loglog(param_counts, losses_by_params, 'o', markersize=10, 
           label='Measured (smaller models)', color='blue')

# 拟合曲线
extended_params = np.logspace(3, 12, 100)
plt.loglog(extended_params, scaling_law_params(extended_params, *params_fit), 
           '--', linewidth=2, label='Power Law Extrapolation', color='blue', alpha=0.5)

# 未来预测
plt.loglog(future_params, predicted_losses, 's', markersize=12, 
           label='Predicted (larger models)', color='red', zorder=5)

# 注释著名模型尺寸
famous_models = [
    (1.5e8, 'GPT-2'),
    (1.75e9, 'GPT-3'),
    (1.75e11, 'GPT-3.5'),
]

for params, name in famous_models:
    loss_pred = scaling_law_params(params, *params_fit)
    plt.plot(params, loss_pred, 'r*', markersize=15)
    plt.annotate(name, (params, loss_pred), 
                xytext=(10, 10), textcoords='offset points', fontsize=10)

plt.xlabel('Number of Parameters (N)')
plt.ylabel('Predicted Loss (L)')
plt.title('Scaling Law Extrapolation to Larger Models')
plt.legend()
plt.grid(True, alpha=0.3, which='both')
plt.show()

print("\nPredicted Performance:")
for N, L in zip(future_params, predicted_losses):
    print(f"  {N:.0e} params → Loss = {L:.4f}")

## 要点总结

### 主要发现（Kaplan 等，2020）：

1. **幂律缩放**：损失与 N、D、C 之间遵循幂律关系
   - L(N) ∝ N^(-α_N)
   - L(D) ∝ D^(-α_D)
   - L(C) ∝ C^(-α_C)

2. **平滑且可预测**：可以跨越 7 个以上数量级进行外推

3. **提前停止**：计算最优的训练会在完全收敛之前停止

4. **迁移性**：缩放定律可以跨任务迁移

### Chinchilla 的发现（Hoffmann 等，2022）：

1. **计算最优**：对于预算 C，应采用
   - N ∝ C^0.5
   - D ∝ C^0.5
   
2. **此前的模型训练不充分**：
   - GPT-3：175B 参数，300B tokens
   - 计算最优方案：70B 参数，1.4T tokens（Chinchilla）

3. **数据量与参数量同样重要**

### 实际意义：

1. **资源分配**：平衡模型规模与训练数据量
2. **性能预测**：在训练前估计最先进水平（SOTA）
3. **研究规划**：判断性能提升将来自哪里
4. **成本优化**：避免参数过多而训练不足

### 缩放定律指数：
- **参数**：α_N ≈ 0.076
- **数据**：α_D ≈ 0.095  
- **计算**：α_C ≈ 0.050

### 为什么会出现幂律？
- 语言底层的统计结构
- 与信息论相一致
- 反映不同规模下的学习难度

### 未来方向：
- 扩展到多模态模型
- 架构创新（MoE 等）
- 数据质量与数量之间的关系
- 大规模模型的涌现能力
